<a href="https://colab.research.google.com/github/JoaoLopesMendes/Projeto-IA-ENEM-2023/blob/main/Fase2_projIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PROJETO IA - 7°J SI - MACKENZIE
# Integrantes: João Pedro Lopes Mendes, Luis Felipe Cunha
#
# FASE 2 (N2): Pipeline de Detecção de Superação Acadêmica e Análise de Resíduos
# ==============================================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. CARREGAMENTO DOS DADOS COM PARÂMETROS CORRETOS
caminho = '/content/MICRODADOS_ENEM_2023.csv'
df = pd.read_csv(caminho, sep=';', encoding='latin-1', nrows=10000)

print(f"Dataset carregado para a N2: {df.shape[0]} registros.")

# [AJUSTE CRÍTICO]: Limpeza e Engenharia de Atributos vindos da N1
# Mantendo apenas alunos presentes em todas as provas
df_validos = df[
    (df['TP_PRESENCA_CN'] == 1) &
    (df['TP_PRESENCA_CH'] == 1) &
    (df['TP_PRESENCA_LC'] == 1) &
    (df['TP_PRESENCA_MT'] == 1)
].copy()

# Criando a variável Target que servirá para a análise de resíduos
colunas_notas = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']
df_validos['NOTA_MEDIA'] = df_validos[colunas_notas].mean(axis=1)

# Removendo quaisquer linhas que fiquem com dados socioeconômicos nulos na amostra
df_validos = df_validos.dropna(subset=['Q006', 'Q001', 'Q002', 'Q025', 'TP_ESCOLA'])

# 2. PRÉ-PROCESSAMENTO: TRANSFORMAÇÃO ORDINAL
categorias_renda = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q']
categorias_estudos = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

encoder = OrdinalEncoder(categories=[categorias_renda, categorias_estudos, categorias_estudos])

# Aplicando o encoder no DataFrame tratado
df_validos[['RENDA_NUM', 'ESTUDOS_PAI_NUM', 'ESTUDOS_MAE_NUM']] = encoder.fit_transform(
    df_validos[['Q006', 'Q001', 'Q002']]
)

# ==============================================================================
# FASE 1: ANÁLISE DE RESÍDUOS (Definição Matemática de Superação)
# ==============================================================================

X_baseline = df_validos[['RENDA_NUM', 'ESTUDOS_PAI_NUM', 'ESTUDOS_MAE_NUM']]
y_baseline = df_validos['NOTA_MEDIA']

# Treinando a Regressão Linear para estipular a "Nota Esperada" por contexto social
modelo_baseline = LinearRegression()
modelo_baseline.fit(X_baseline, y_baseline)

# Calculando a predição da baseline e os resíduos individuais
df_validos['NOTA_ESPERADA'] = modelo_baseline.predict(X_baseline)
df_validos['RESIDUO'] = df_validos['NOTA_MEDIA'] - df_validos['NOTA_ESPERADA']

# Definindo o corte estatístico para "Superação Acadêmica" (+1.5 Desvios Padrão)
limite_superacao = df_validos['RESIDUO'].mean() + (1.5 * df_validos['RESIDUO'].std())
df_validos['SUPERACAO_ACADEMICA'] = (df_validos['RESIDUO'] > limite_superacao).astype(int)

print(f"Alunos filtrados por presença: {df_validos.shape[0]}")
print(f"Casos de Superação Acadêmica detectados: {df_validos['SUPERACAO_ACADEMICA'].sum()} alunos.")

# Visualização dos Resíduos e do Limite de Superação
plt.figure(figsize=(10, 5))
sns.histplot(data=df_validos, x='RESIDUO', hue='SUPERACAO_ACADEMICA', multiple='stack', palette='viridis', bins=50)
plt.axvline(limite_superacao, color='red', linestyle='--', label=f'Corte de Superação (+1.5 std): {limite_superacao:.2f}')
plt.title('Distribuição dos Resíduos e Identificação de Outliers Positivos')
plt.xlabel('Resíduo (Nota Real - Nota Esperada pela Renda)')
plt.ylabel('Contagem de Alunos')
plt.legend()
plt.tight_layout()
plt.show()

# ==============================================================================
# FASE 2: CLASSIFICAÇÃO E INFERÊNCIA DE FATORES (Machine Learning)
# ==============================================================================

# Mapeando variáveis preditoras para o classificador avançado
df_validos['INTERNET_NUM'] = df_validos['Q025'].map({'A': 0, 'B': 1})

features_inferencia = ['INTERNET_NUM', 'TP_ESCOLA']
X_ia = df_validos[features_inferencia]
y_ia = df_validos['SUPERACAO_ACADEMICA']

# Divisão de Treino e Teste (80/20) com estratificação devido ao desbalanceamento do target
X_train, X_test, y_train, y_test = train_test_split(X_ia, y_ia, test_size=0.2, random_state=42, stratify=y_ia)

# Treinamento da Random Forest com balanceamento de pesos para compensar a minoria
modelo_ia = RandomForestClassifier(random_state=42, class_weight='balanced')
modelo_ia.fit(X_train, y_train)

# 3. AVALIAÇÃO DO MODELO
y_pred = modelo_ia.predict(X_test)
print("\n=== MÉTRICAS DE AVALIAÇÃO DO MODELO DE IA ===")
print(classification_report(y_test, y_pred))

# Extração de Feature Importance
importâncias = modelo_ia.feature_importances_
print("=== IMPORTÂNCIA DOS ATRIBUTOS NA SUPERAÇÃO ===")
for feat, imp in zip(features_inferencia, importâncias):
    print(f"Influência do atributo [{feat}]: {imp*100:.2f}%")

FileNotFoundError: [Errno 2] No such file or directory: '/content/MICRODADOS_ENEM_2023.csv'